<a href="https://colab.research.google.com/github/Higashi88jp/Matsuo_Lab/blob/main/DL_Basic_2026_Spring_Competition_NYUv2_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning 基礎講座　最終課題: NYUv2 セマンティックセグメンテーション

## 概要
RGB画像から、画像内の各ピクセルがどのクラスに属するかを予測するセマンティックセグメンテーションタスク.

### データセット
- データセット: NYUv2 dataset
- 訓練データ: 795枚
- テストデータ: 654枚
- 入力: RGB画像 + 深度マップ（元画像サイズは可変）
- 出力: 13クラスのセグメンテーションマップ
- 評価指標: Mean IoU (Intersection over Union)

### データセットの詳細（[NYU Depth Dataset V2](https://cs.nyu.edu/~fergus/datasets/nyu_depth_v2.html)）
- 画像は屋内シーンを撮影したもので、家具や壁、床などの物体が含まれています.
- 各画像に対して13クラスのセグメンテーションラベルが提供されます.
- データは以下のディレクトリ構造で提供:
```
data/NYUv2/
├─train/
│  ├─image/      # RGB画像
│  │    000000.png
│  │    ...
│  │
│  ├─depth/      # 深度マップ
│  │    000000.png
│  │    ...
│  │
│  └─label/      # 13クラスセグメンテーション（教師ラベル）
│       000000.png
│       ...
└─test/
   ├─image/      # RGB画像
   │    000000.png
   │    ...
   │  ├─depth/   # 深度マップ
   │    000000.png
   │    ...
```

### タスクの詳細
- 入力のRGB画像と深度マップから、各ピクセルが13クラスのどれに属するかを予測するタスクです.
- 評価はMean IoUを使用します．
  - 各クラスごとにIoUを計算し、その平均を取ります.
  - IoUは以下の式で計算:
  $$IoU = \frac{TP}{TP + FP + FN}$$
    - TP: True Positive（正しく予測されたピクセル数）
    - FP: False Positive（誤って予測されたピクセル数）
    - FN: False Negative（見逃したピクセル数）

### 前処理
- 入力画像は512×512にリサイズされます.
- ピクセル値は0-1に正規化されます.
- セグメンテーションラベルは0-12の整数値（13クラス）です．
  - 255はignore index（評価から除外）

### 提出形式
- テスト画像（RGB + Depth）の各ピクセルに対してクラス（0~12）を予測したものをnumpy配列として保存されます.
- ファイル名: `submission.npy`
- 配列の形状: [テストデータ数, 高さ, 幅]
- 各ピクセルの値: 0-12の整数（予測クラス）



## 考えられる工夫の例
- 事前学習モデルの fine-tuning
    - ImageNetなどで事前学習されたモデルを本データセットでfine-tuningすることで性能向上が見込めます.
- 損失関数の再設計
    - クラスごとの出現頻度に応じて損失を補正するように損失関数を設計すると、クラス分布の不均衡に対してロバストな学習ができます.
- 画像の前処理
    - RandomResizedCrop / Flip / ColorJitter 等のデータ拡張を追加することで，汎化性能の向上が見込めます．

## 修了要件を満たす条件
- ベースラインでは，omnicampus 上での性能評価において， 36.4% となります．したがって，ベースラインである 36.4% を超えた提出のみ，修了要件として認めます．
- ベースラインから改善を加えることで， 50%以上に性能向上することを運営で確認しています．こちらを 1つの指標として取り組んでみてください．

## 注意点
- 最終的な予測モデルは，**配布している訓練データを用いて学習**（ファインチューニング含む）したものとしてください．
- 学習を行わず，**事前学習済みモデルの知識のみを利用した推論は禁止**します．  
（例: ChatGPT 等の LLM に入力して推論を得るのみ）

### 事前学習モデルの利用
許可される事項
- **構成要素としての事前学習モデルの利用**: 自身で実装したアーキテクチャの一部（特徴抽出，埋め込みなど）として事前学習モデル（BERT，ViT など）を利用することは可能です．
- **ファインチューニング**: 上記の用途で利用している事前学習モデルのファインチューニングは可能です．

禁止される事項  
- **タスク解決用の事前学習モデルの利用**: transformers などで提供されている，対象タスクを直接解くための事前学習モデルでそのまま推論のみ，またはファインチューニングのみで利用することは禁止とします．
  - 禁止事項の例: VQA タスクを直接解くための事前学習モデルを VQA タスクで利用する．

### データの準備
データをダウンロードした際に，google drive したため，利用するために google drive をマウントする必要があります．また， drive 上で展開することができないため，/content ディレクトリ下にコピーし "data.zip" を展開します．  
google drive 上に "data.zip" が配置されていない場合は実行できません．google drive 上に "data.zip" (**831MB**) を配置することが可能であれば，"data_download.ipynb" を先に実行してください．難しい場合は，omnicampus 演習環境を利用してください．．



In [ ]:
# omnicampus 上では 4 セル目まで実行不要
# ドライブのマウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# データダウンロード用の notebook にてgoogle drive への保存後，
# 反映に時間がかかる可能性がありますので，google drive のマウント後，
# data.zip がディレクトリ内にあることを確認してから実行してください．
# data.zip を /content 下にコピーする
!cp "/content/drive/MyDrive/Colab Notebooks/DLBasics2026/Competition/NYUv2/data.zip" "/content"

In [ ]:
# カレントディレクトリ下のファイル群を確認
# data.zip が表示されれば問題ないです
%ls

In [ ]:
# データを解凍する
!unzip data.zip
!mkdir data
!mv train test data/

omnicampus 演習環境では，`gsutil`を利用してデータをダウンロードすることができます．ダウンロード後，data ディレクトリが存在するディレクトリをカレントディレクトリとするだけで良いです．



In [ ]:
# omnicampus 実行用
# 以下の例では/workspace/Segmentation/split_data_scripts/omnicampus に data ディレクトリがあると想定
# %cd /workspace/Segmentation/split_data_scripts_omnicampus

In [ ]:
# omnicampus 実行用
# !pip install numpy==1.22.2 h5py scikit-image

# import library

In [ ]:
!pip install segmentation-models-pytorch -q

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
import os
import random
import time
from tqdm import tqdm
import numpy as np
from scipy.io import loadmat
from PIL import Image
import torch
import torch.nn as nn
from torch import optim
import torch.utils.data as data
from torch.utils.data import random_split, DataLoader
from torchvision.datasets import VisionDataset
from torchvision import transforms
from torchvision.transforms import (
    Compose,
    RandomResizedCrop,
    RandomHorizontalFlip,
    ColorJitter,
    GaussianBlur,
    Resize,
    ToTensor,
    Normalize,
    Lambda,
    InterpolationMode
)
from torch.amp import autocast, GradScaler
from dataclasses import dataclass
import torchvision.transforms.functional as TF
import torch.nn.functional as F


# DataLoader

In [ ]:
# カラーマップ生成関数：セグメンテーションの可視化用
def colormap(N=256, normalized=False):
    def bitget(byteval, idx):
        return ((byteval & (1 << idx)) != 0)

    dtype = 'float32' if normalized else 'uint8'
    cmap = np.zeros((N, 3), dtype=dtype)
    for i in range(N):
        r = g = b = 0
        c = i
        for j in range(8):
            r = r | (bitget(c, 0) << 7-j)
            g = g | (bitget(c, 1) << 7-j)
            b = b | (bitget(c, 2) << 7-j)
            c = c >> 3

        cmap[i] = np.array([r, g, b])

    cmap = cmap/255 if normalized else cmap
    return cmap

# NYUv2データセット：RGB画像、セグメンテーション、深度、法線マップを提供するデータセット
class NYUv2(VisionDataset):
    cmap = colormap()
    def __init__(self,
                 root,
                 split='train',
                 include_depth=False,
                 transform=None,          # RGB画像用（Resize+ToTensor+Normalize）
                 depth_transform=None,    # Depth用（Resize+ToTensorのみ、Normalizeなし）
                 target_transform=None,
                 augment=None,
                 crop_size=None,          # RandomResizedCropの出力サイズ
                 crop_scale=(0.5, 1.0),
                 crop_ratio=(0.75, 1.3333),
                 color_jitter=None,       # transforms.ColorJitter インスタンス（RGBのみに適用）
                 ):
        super(NYUv2, self).__init__(root, transform=transform, target_transform=target_transform)

        assert(split in ('train', 'test'))
        self.root = root
        self.split = split
        self.include_depth = include_depth
        self.augment = (split == 'train') if augment is None else augment
        self.train_idx = np.array([255, ] + list(range(13)))

        # depth用transformが未指定ならtransformと同じものを使う（後方互換）
        self.depth_transform = depth_transform if depth_transform is not None else transform

        self.crop_size = crop_size
        self.crop_scale = crop_scale
        self.crop_ratio = crop_ratio
        self.color_jitter = color_jitter

        img_names = os.listdir(os.path.join(self.root, self.split, 'image'))
        img_names.sort()
        images_dir = os.path.join(self.root, self.split, 'image')
        self.images = [os.path.join(images_dir, name) for name in img_names]

        label_dir = os.path.join(self.root, self.split, 'label')
        if (self.split == 'train'):
          self.labels = [os.path.join(label_dir, name) for name in img_names]
          self.targets = self.labels

        depth_dir = os.path.join(self.root, self.split, 'depth')
        self.depths = [os.path.join(depth_dir, name) for name in img_names]

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert("RGB")
        depth = Image.open(self.depths[idx])

        if self.split == 'train' and self.target_transform is not None:
            target = Image.open(self.targets[idx])

        # ------------------
        #   データ拡張（学習時のみ）
        #   乱数を1回だけ決めて、image / depth / target に同じ変換を同期して適用
        # ------------------
        if self.split == 'train' and self.augment:

            # --- RandomResizedCrop ---
            if self.crop_size is not None:
                i, j, h, w = RandomResizedCrop.get_params(
                    image, scale=self.crop_scale, ratio=self.crop_ratio
                )
                image = TF.resized_crop(image, i, j, h, w, size=self.crop_size,
                                         interpolation=InterpolationMode.BILINEAR)
                depth = TF.resized_crop(depth, i, j, h, w, size=self.crop_size,
                                         interpolation=InterpolationMode.BILINEAR)
                if self.target_transform is not None:
                    target = TF.resized_crop(target, i, j, h, w, size=self.crop_size,
                                              interpolation=InterpolationMode.NEAREST)

            # --- 水平flip ---
            if random.random() < 0.5:
                image = TF.hflip(image)
                depth = TF.hflip(depth)
                if self.target_transform is not None:
                    target = TF.hflip(target)

            # --- ColorJitter（RGB画像のみ。depth/targetには適用しない） ---
            if self.color_jitter is not None:
                image = self.color_jitter(image)

        # ------------------
        #   共通の変形（Resize/ToTensor/Normalizeなど）
        # ------------------
        if self.transform is not None:
            image = self.transform(image)
        if self.depth_transform is not None:
            depth = self.depth_transform(depth)

        if self.split == 'train' and self.target_transform is not None:
            target = self.target_transform(target)

        if self.split == 'test':
            if self.include_depth:
                return image, depth
            return image
        else:
            if self.include_depth:
                return image, depth, target
            return image, target

    def __len__(self):
        return len(self.images)


# Model Section


In [ ]:
# 2つの畳み込み層とバッチ正規化、ReLUを含むブロック
# UNetの各層で使用される基本的な畳み込みブロック
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.double_conv(x)

# UNetモデル：エンコーダ・デコーダ構造のセグメンテーションモデル
class UNet(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        # エンコーダ部分：特徴量の抽出と空間サイズの縮小
        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)
        self.pool = nn.MaxPool2d(2)

        # デコーダ部分：特徴量の統合と空間サイズの復元
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec3 = DoubleConv(512 + 256, 256)
        self.dec2 = DoubleConv(256 + 128, 128)
        self.dec1 = DoubleConv(128 + 64, 64)

        # 最終層：クラス数に応じた出力チャネルに変換
        self.final = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # エンコーダパス：特徴抽出とダウンサンプリング
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # デコーダパス：特徴統合とアップサンプリング（スキップ接続を使用）
        d3 = self.dec3(torch.cat([self.up(e4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up(d2), e1], dim=1))

        return self.final(d1)


# Train and Valid

In [ ]:
@dataclass
class TrainingConfig:
    # データセットパス
    dataset_root: str = "data"

    # データ関連
    batch_size: int = 32
    num_workers: int = 4

    # モデル関連
    in_channels: int = 3
    num_classes: int = 13  # NYUv2データセットの場合

    # 学習関連
    epochs: int = 100
    learning_rate: float = 0.001
    weight_decay: float = 1e-4

    # データ分割関連
    train_val_split: float = 0.8  # 訓練データの割合

    # デバイス設定
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    # チェックポイント関連（Google Drive上に保存）
    checkpoint_dir: str = "/content/drive/MyDrive/Colab Notebooks/DLBasics2026/Competition/NYUv2/checkpoints_resnet34"
    save_interval: int = 5  # エポックごとのモデル保存間隔

    # データ拡張・前処理関連
    image_size: tuple = (320, 256)
    normalize_mean: tuple = (0.485, 0.456, 0.406)  # ImageNetの標準化パラメータ
    normalize_std: tuple = (0.229, 0.224, 0.225)

    def __post_init__(self):
        import os
        os.makedirs(self.checkpoint_dir, exist_ok=True)


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
import os
import json
import torch

class TopKCheckpointManager:
    """
    val mIoU 上位K個のチェックポイントのみを保持するマネージャ。
    Drive上のディレクトリに状態をJSONで保存するので、クラッシュ後も再開可能。
    """
    def __init__(self, save_dir, k=6, state_filename="topk_state.json"):
        self.save_dir = save_dir
        self.k = k
        self.state_path = os.path.join(save_dir, state_filename)
        os.makedirs(save_dir, exist_ok=True)
        self.checkpoints = []
        self._load_state()

    def _load_state(self):
        if os.path.exists(self.state_path):
            with open(self.state_path, "r") as f:
                data = json.load(f)
            self.checkpoints = [
                (c["val_miou"], c["epoch"], c["filename"])
                for c in data
                if os.path.exists(os.path.join(self.save_dir, c["filename"]))
            ]
            self.checkpoints.sort(key=lambda x: -x[0])
            if self.checkpoints:
                print(f"[TopK] 既存状態を復元: {len(self.checkpoints)}件 "
                      f"(最高 epoch={self.checkpoints[0][1]} mIoU={self.checkpoints[0][0]:.4f})")

    def _save_state(self):
        data = [
            {"val_miou": miou, "epoch": ep, "filename": fn}
            for miou, ep, fn in self.checkpoints
        ]
        with open(self.state_path, "w") as f:
            json.dump(data, f, indent=2)

    def maybe_save(self, model, epoch, val_miou):
        if len(self.checkpoints) < self.k or val_miou > self.checkpoints[-1][0]:
            filename = f"epoch{epoch:03d}.pt"
            save_path = os.path.join(self.save_dir, filename)
            torch.save(model.state_dict(), save_path)

            self.checkpoints.append((val_miou, epoch, filename))
            self.checkpoints.sort(key=lambda x: -x[0])

            dropped = None
            if len(self.checkpoints) > self.k:
                dropped = self.checkpoints.pop()
                drop_path = os.path.join(self.save_dir, dropped[2])
                if os.path.exists(drop_path):
                    os.remove(drop_path)

            self._save_state()

            msg = f"[TopK] epoch{epoch:03d} (mIoU={val_miou:.4f}) を保存"
            if dropped:
                msg += f" / epoch{dropped[1]:03d} (mIoU={dropped[0]:.4f}) を削除"
            print(msg)
            return True
        return False

    def get_paths(self):
        return [os.path.join(self.save_dir, fn) for _, _, fn in self.checkpoints]

    def summary(self):
        print("=== Top-K Checkpoints ===")
        for miou, ep, fn in self.checkpoints:
            print(f"  epoch{ep:03d}  val_mIoU={miou:.4f}  ({fn})")


In [36]:
set_seed(42)

# 設定の初期化
config = TrainingConfig(
    dataset_root='/content/data',
    batch_size=16,
    num_workers=4,
    learning_rate=1e-4,
    epochs=100,
    image_size=(320, 256),   # ← 240 → 256
    in_channels=4  # RGB(3チャネル) + Depth(1チャネル)
)

# ------------------
#    Dataloader
# ------------------

# ------------------
#   Transform定義
# ------------------

# RGB画像用：Resize → ToTensor → Normalize
# ※ RandomResizedCrop/Flip/ColorJitterはデータセット内部で同期処理するため、
#    ここには含めない（学習時はcrop_sizeで既に目的サイズになっているため、
#    このResizeは検証/テスト時にのみ実質的に効く）
transform_img = Compose([
    Resize(config.image_size, interpolation=InterpolationMode.BILINEAR),
    ToTensor(),
    Normalize(mean=config.normalize_mean, std=config.normalize_std),
])

# Depth用：Resize → ToTensor のみ（RGB統計量でのNormalizeは不適切なため適用しない）
transform_depth = Compose([
    Resize(config.image_size, interpolation=InterpolationMode.BILINEAR),
    ToTensor(),
])

# セグメンテーションラベルのTransform：リサイズとテンソル変換
target_transform = Compose([
    Resize(config.image_size, interpolation=InterpolationMode.NEAREST),
    Lambda(lambda lbl: torch.from_numpy(np.array(lbl)).long())
])

# 学習時のみ使うColorJitter（RGB画像のみに適用）
train_color_jitter = ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)

# データセットの準備
train_dataset = NYUv2(
    root=config.dataset_root, split='train', include_depth=True,
    transform=transform_img, depth_transform=transform_depth, target_transform=target_transform,
    augment=True,
    crop_size=config.image_size,
    crop_scale=(0.5, 1.0),
    color_jitter=train_color_jitter,
)
val_dataset_noaug = NYUv2(
    root=config.dataset_root, split='train', include_depth=True,
    transform=transform_img, depth_transform=transform_depth, target_transform=target_transform,
    augment=False  # ここがポイント：crop/flip/jitterなし、Resizeのみ
)

n_total = len(train_dataset)
n_train = int(n_total * config.train_val_split)
n_val = n_total - n_train

# インデックスだけ共有して、データセット本体は別々にする
train_idx_subset, val_idx_subset = random_split(
    range(n_total), [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

train_subset = torch.utils.data.Subset(train_dataset, list(train_idx_subset))
val_subset = torch.utils.data.Subset(val_dataset_noaug, list(val_idx_subset))

train_data = DataLoader(train_subset, batch_size=config.batch_size, shuffle=True, num_workers=config.num_workers)
val_data = DataLoader(val_subset, batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers)

# テストデータセット
test_dataset = NYUv2(
    root=config.dataset_root,
    split='test',
    include_depth=True,
    transform=transform_img,
    depth_transform=transform_depth,
)

test_data = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=config.num_workers)

# モデルとトレーニングの設定
device = config.device
print(f"Using device: {device}")

# ------------------
#    Model
# ------------------
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="resnet34",        # 軽くて扱いやすいバックボーン
    encoder_weights="imagenet",     # ImageNet事前学習済み重みを使う
    in_channels=config.in_channels, # 4 (RGB+Depth) のまま使える
    classes=config.num_classes,     # 13
).to(device)

print("Computing class weights from training labels...")
class_counts = torch.zeros(config.num_classes)

count_loader = DataLoader(train_dataset, batch_size=1, num_workers=2)
for _, _, label in tqdm(count_loader):
    valid = label != 255
    for c in range(config.num_classes):
        class_counts[c] += (label[valid] == c).sum()

print("class_counts:", class_counts)

class_weights = 1.0 / (class_counts + 1e-6)
class_weights = class_weights / class_weights.sum() * config.num_classes
class_weights = class_weights.to(device)
print("class_weights:", class_weights)

# ------------------
#    optimizer
# ------------------
optimizer = optim.Adam([
    {"params": model.encoder.parameters(), "lr": 1e-5},          # 事前学習済み部分は小さめの学習率
    {"params": model.decoder.parameters(), "lr": config.learning_rate},
    {"params": model.segmentation_head.parameters(), "lr": config.learning_rate},
], weight_decay=config.weight_decay)

criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=255)

# ------------------
#    Training
# ------------------
num_epochs = config.epochs
scaler = GradScaler('cuda')

best_val_loss = float('inf')  # 追加：学習ループの直前に
best_val_miou = -1.0
miou_history = []

# ------------------
#    Top-K Checkpoint Manager
# ------------------
ckpt_manager = TopKCheckpointManager(
    save_dir=config.checkpoint_dir,
    k=6
)

model.train()
for epoch in range(num_epochs):
    total_loss = 0
    print(f"on epoch: {epoch+1}")
    with tqdm(train_data) as pbar:
        for batch_idx, (image, depth, label) in enumerate(pbar):
            image, depth, label = image.to(device), depth.to(device), label.to(device)
            optimizer.zero_grad()

            with autocast('cuda'):
              x = torch.cat((image, depth), dim=1) # RGB + Depth
              pred = model(x)
              loss = criterion(pred, label)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            del image, depth, label, pred, loss

    print(f'Epoch {epoch+1}, Loss: {total_loss / len(train_data)}')

    # ------------------
    #   検証（loss + mIoU を同時に計算）
    # ------------------
    model.eval()
    val_loss = 0.0
    intersection = torch.zeros(config.num_classes)
    union = torch.zeros(config.num_classes)
    with torch.no_grad():
        for image, depth, label in val_data:
            image, depth, label = image.to(device), depth.to(device), label.to(device)
            with autocast('cuda'):
                x = torch.cat((image, depth), dim=1)
                pred_logits = model(x)
                loss = criterion(pred_logits, label)
            val_loss += loss.item()

            pred = pred_logits.argmax(dim=1)
            mask = label != 255
            for cls in range(config.num_classes):
                p = (pred == cls) & mask
                t = (label == cls) & mask
                intersection[cls] += (p & t).sum().item()
                union[cls] += (p | t).sum().item()

    val_loss /= len(val_data)
    valid = union > 0
    val_miou = (intersection[valid] / union[valid]).mean().item()
    print(f'  -> val loss: {val_loss:.4f}, val mIoU: {val_miou:.4f}')

    miou_history.append({'epoch': epoch + 1, 'val_loss': val_loss, 'val_miou': val_miou})

    import json
    with open(os.path.join(config.checkpoint_dir, "miou_history.json"), "w") as f:
        json.dump(miou_history, f, indent=2)

    # Top-K保存（5の倍数に関わらず、毎エポックval_miouで判定）
    ckpt_manager.maybe_save(model, epoch + 1, val_miou)

    # ベスト保存（lossとmIoU両方残しておく。最終的にはmIoU基準を使う）
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), os.path.join(config.checkpoint_dir, "best_loss.pt"))
    if val_miou > best_val_miou:
        best_val_miou = val_miou
        torch.save(model.state_dict(), os.path.join(config.checkpoint_dir, "best_miou.pt"))

    model.train()

ckpt_manager.summary()  # 最終結果の確認用

# モデルの保存
current_time = time.strftime("%Y%m%d%H%M%S")
model_path = f"model_{current_time}.pt"
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Using device: cpu
Computing class weights from training labels...


100%|██████████| 795/795 [01:02<00:00, 12.73it/s]
/tmp/ipykernel_2139/3694950712.py:136: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = GradScaler('cuda')


class_counts: tensor([ 2424477.,   431614.,   831609.,  2497234.,  6220203., 10562462.,
         9151792.,  1331979.,  1715979.,  2281206.,   432493., 15460097.,
         3169637.])
class_weights: tensor([0.5853, 3.2877, 1.7064, 0.5682, 0.2281, 0.1343, 0.1551, 1.0654, 0.8269,
        0.6221, 3.2810, 0.0918, 0.4477])
on epoch: 1


  0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipykernel_2139/3694950712.py:159: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with autocast('cuda'):
 18%|█▊        | 7/40 [03:28<16:24, 29.82s/it]


KeyboardInterrupt: 

In [ ]:
import json
import pandas as pd

try:
    history = miou_history
    print("メモリ上のmiou_historyを使用")
except NameError:
    history_path = os.path.join(config.checkpoint_dir, "miou_history.json")
    with open(history_path, "r") as f:
        history = json.load(f)
    print(f"Driveのjsonから読み込み: {history_path}")

ckpt_paths = ckpt_manager.get_paths()
print("アンサンブルに使うcheckpoint:", ckpt_paths)


In [ ]:
import gc

NUM_TEST = len(test_dataset)
H, W = 512, 512
num_classes = config.num_classes

all_probs = torch.zeros((NUM_TEST, num_classes, H, W), dtype=torch.float16)

with torch.no_grad():
    for ckpt_path in ckpt_paths:
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        model.eval()
        idx = 0
        for image, depth in tqdm(test_data, desc=os.path.basename(ckpt_path)):
            bs = image.size(0)
            image, depth = image.to(device), depth.to(device)
            x = torch.cat((image, depth), dim=1)
            with autocast('cuda'):
                output = model(x)
                output_512 = F.interpolate(output, size=(H, W), mode='bilinear', align_corners=False)
                prob = F.softmax(output_512, dim=1).float()
            all_probs[idx:idx+bs] += prob.cpu().half()
            idx += bs
            del image, depth, x, output, output_512, prob
        torch.cuda.empty_cache()
        gc.collect()

all_probs /= len(ckpt_paths)
predictions = all_probs.argmax(dim=1).numpy().astype(np.uint8)
np.save('submission.npy', predictions)
print("Predictions saved to submission.npy")
print("shape:", predictions.shape)

del all_probs
gc.collect()


## 提出方法

以下の3点をzip化し，Omnicampusの「最終課題 (セグメンテーション)」から提出してください．

- `submission.npy`
- `model.pt`や`model_best.pt`など，テストに使用した重み（拡張子は`.pt`のみ）
- 本Colab Notebook

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
import os

notebook_path = "/content/drive/MyDrive/Colab Notebooks/DLBasics2026/Competition/NYUv2/DL_Basic_2026_Spring_Competition_NYUv2_baseline.ipynb"

# 追加：存在確認。ここで止まったら、ファイル名・パスをDrive上で直接確認してください
if not os.path.exists(notebook_path):
    raise FileNotFoundError(
        f"ノートブックが見つかりません: {notebook_path}\n"
        "Google Drive上でファイル名・保存場所を確認してください（リネームしていないか、最新版が同期されているか）。"
    )
if not os.path.exists(model_path):
    raise FileNotFoundError(f"モデルファイルが見つかりません: {model_path}")
if not os.path.exists("submission.npy"):
    raise FileNotFoundError("submission.npy が見つかりません。Evaluationセルを先に実行してください。")

with ZipFile("submission.zip",
             mode="w",
             compression=ZIP_DEFLATED,
             compresslevel=9) as zf:
    zf.write("submission.npy")
    zf.write(model_path)
    zf.write(notebook_path,
             arcname="DL_Basic_2026_Spring_Competition_NYUv2_baseline.ipynb")

from google.colab import files
!cp submission.zip "/content/drive/MyDrive/Colab Notebooks/DLBasics2026/Competition/NYUv2/"
files.download("submission.zip")


In [ ]:
import os
print("submission.npy:", os.path.getsize("submission.npy") / 1e6, "MB")
print("model_path:", os.path.getsize(model_path) / 1e6, "MB")